# **Sentiment Analysis** - IMDB Dataset

Contains next sections:
- **Section 1:** Imports
- **Section 2:** Loading and preparing data
- **Section 3:** Split data
- **Section 4:** Algorithm 1
- **Section 5:** Algorithm 2
- **Section 6:** PCA analysis
- **Section 7:** Final testing on an independent test set
- **Section 8:** Data visualization
- **Section 9:** Final results and saving the model

## **Section 1:** Imports

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                           classification_report, confusion_matrix, roc_curve, auc)
from sklearn.preprocessing import StandardScaler
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from nltk import FreqDist
import matplotlib.pyplot as plt
import seaborn as sns
import utils  # tvoj utils modul

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\iam0v\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\iam0v\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\iam0v\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## **Section 2:** Loading and preparing data

In [3]:
print("="*50)
print("SENTIMENT ANALYSIS - IMDB Dataset")
print("="*50)

# Učitavanje podataka
texts = []
labels = []
directory = r"C:\Users\iam0v\Downloads\datasets\imdb\train\pos"
utils.load_data(directory, texts, labels)
directory = r"C:\Users\iam0v\Downloads\datasets\imdb\train\neg"
utils.load_data(directory, texts, labels)
directory = r"C:\Users\iam0v\Downloads\datasets\imdb\test\pos"
utils.load_data(directory, texts, labels)
directory = r"C:\Users\iam0v\Downloads\datasets\imdb\test\neg"
utils.load_data(directory, texts, labels)

# Preprocesiranje
texts = [utils.lemmatize(utils.remove_stopwords(utils.remove_breaklines(text))) for text in texts]
texts = [utils.clean_text(text) for text in texts]

print(f"Ukupno tekstova: {len(texts)}")
print(f"Ukupno labela: {len(labels)}")

SENTIMENT ANALYSIS - IMDB Dataset
Ukupno tekstova: 50000
Ukupno labela: 50000


## **Section 3:** Split data

In [4]:
# VAŽNO: Izdvajanje finalnog test skupa (15%) koji se ne koristi do kraja
X_temp, X_final_test, y_temp, y_final_test = train_test_split(
    texts, labels, test_size=0.15, random_state=42, stratify=labels
)

# Podela preostalog na train i validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

print(f"Train set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")
print(f"Final test set: {len(X_final_test)} samples (izdvojen za finalno testiranje)")

Train set: 34000 samples
Validation set: 8500 samples
Final test set: 7500 samples (izdvojen za finalno testiranje)


## **Section 4:** Algorithm 1

In [5]:
print("\n--- Algoritam 1: LSTM ---")

# Priprema tokenizera
fdist = FreqDist()
for text in X_train:
    for word in text.split():
        fdist[word.lower()] += 1

num_of_words = len([word for word in fdist if fdist[word] > 3])
print(f"Broj reči u rečniku: {num_of_words}")

tokenizer_lstm = Tokenizer(num_words=num_of_words,
                          filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
                          lower=True,
                          split=' ')
tokenizer_lstm.fit_on_texts(X_train)


--- Algoritam 1: LSTM ---
Broj reči u rečniku: 33949


In [6]:
# Sekvence
X_train_seq = tokenizer_lstm.texts_to_sequences(X_train)
X_val_seq = tokenizer_lstm.texts_to_sequences(X_val)
X_final_seq = tokenizer_lstm.texts_to_sequences(X_final_test)

# Padding
seq_lengths = [len(text.split()) for text in X_train]
max_length = max(seq_lengths)
print(f"Maksimalna dužina sekvence: {max_length}")

X_train_seq = pad_sequences(X_train_seq, max_length)
X_val_seq = pad_sequences(X_val_seq, max_length)
X_final_seq = pad_sequences(X_final_seq, max_length)

print(f"Shape training sekvenci: {X_train_seq.shape}")

Maksimalna dužina sekvence: 1545
Shape training sekvenci: (34000, 1545)


In [7]:
# Funkcija za kreiranje LSTM modela sa različitim hiperparametrima
def create_lstm_model(lstm_units1=32, lstm_units2=16, embedding_dim=10):
    model = Sequential()
    model.add(Embedding(num_of_words, embedding_dim))
    model.add(LSTM(lstm_units1, return_sequences=True))
    model.add(LSTM(lstm_units2))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Kreiranje i treniranje najbolje LSTM konfiguracije
print("Treniranje LSTM modela...")
best_lstm = create_lstm_model(32, 16, 10)

# Callback funkcije
early_stopping = EarlyStopping(monitor='val_loss', patience=4, mode='min', verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=0.001, mode='min', verbose=1)
checkpoint = ModelCheckpoint('sentiment_lstm_model.keras', monitor='val_loss', mode='min', save_best_only=True, verbose=1)

# Treniranje
history_lstm = best_lstm.fit(X_train_seq, np.array(y_train), 
                            epochs=10, batch_size=64, 
                            validation_data=(X_val_seq, np.array(y_val)),
                            callbacks=[early_stopping, reduce_lr, checkpoint],
                            verbose=1)

Treniranje LSTM modela...
Epoch 1/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 0s 494ms/step - accuracy: 0.7082 - loss: 0.5181
Epoch 1: val_loss improved from None to 0.32430, saving model to sentiment_lstm_model.keras
532/532 ━━━━━━━━━━━━━━━━━━━━ 283s 528ms/step - accuracy: 0.7982 - loss: 0.4217 - val_accuracy: 0.8687 - val_loss: 0.3243 - learning_rate: 0.0010
Epoch 2/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 0s 499ms/step - accuracy: 0.9182 - loss: 0.2198
Epoch 2: val_loss improved from 0.32430 to 0.28789, saving model to sentiment_lstm_model.keras
532/532 ━━━━━━━━━━━━━━━━━━━━ 296s 556ms/step - accuracy: 0.9165 - loss: 0.2225 - val_accuracy: 0.8845 - val_loss: 0.2879 - learning_rate: 0.0010
Epoch 3/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 0s 692ms/step - accuracy: 0.9524 - loss: 0.1407
Epoch 3: val_loss did not improve from 0.28789
532/532 ━━━━━━━━━━━━━━━━━━━━ 398s 747ms/step - accuracy: 0.9481 - loss: 0.1486 - val_accuracy: 0.8822 - val_loss: 0.3265 - learning_rate: 0.0010
Epoch 4/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 0

## **Section 5:** Algorithm 2

In [8]:
print("\n--- Algoritam 2: SVM sa TF-IDF ---")

# TF-IDF vektorizacija
tfidf_sentiment = TfidfVectorizer(max_features=5000, stop_words='english', max_df=0.7)
X_train_tfidf = tfidf_sentiment.fit_transform(X_train)
X_val_tfidf = tfidf_sentiment.transform(X_val)
X_final_tfidf = tfidf_sentiment.transform(X_final_test)

print(f"TF-IDF shape: {X_train_tfidf.shape}")


--- Algoritam 2: SVM sa TF-IDF ---
TF-IDF shape: (34000, 5000)


In [ ]:
# Grid Search za SVM
print("Grid Search za SVM hiperparametre...")
param_grid_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

svm_classifier = SVC(probability=True, random_state=42)
grid_search_svm = GridSearchCV(svm_classifier, param_grid_svm, 
                               cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search_svm.fit(X_train_tfidf, y_train)

print(f"Najbolji SVM parametri: {grid_search_svm.best_params_}")
print(f"Najbolji CV score: {grid_search_svm.best_score_:.4f}")

best_svm = grid_search_svm.best_estimator_

Grid Search za SVM hiperparametre...
Fitting 5 folds for each of 12 candidates, totalling 60 fits


## **Section 6:** PCA analysis

In [ ]:
print("\n--- Primena PCA na TF-IDF features ---")

# Standardizacija pre PCA (važno za PCA)
scaler = StandardScaler(with_mean=False)  # with_mean=False jer je sparse matrix
X_train_scaled = scaler.fit_transform(X_train_tfidf)
X_val_scaled = scaler.transform(X_val_tfidf)

# PCA sa očuvanjem 95% varijanse
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled.toarray())
X_val_pca = pca.transform(X_val_scaled.toarray())

print(f"Originalna dimenzionalnost: {X_train_tfidf.shape[1]}")
print(f"PCA dimenzionalnost: {X_train_pca.shape[1]}")
print(f"Objašnjena varijansa: {pca.explained_variance_ratio_.sum():.4f}")

In [ ]:
# Grid Search za SVM sa PCA
print("\nGrid Search za SVM sa PCA features...")
svm_pca = SVC(probability=True, random_state=42)
grid_search_svm_pca = GridSearchCV(svm_pca, param_grid_svm, 
                                   cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search_svm_pca.fit(X_train_pca, y_train)

print(f"Najbolji SVM-PCA parametri: {grid_search_svm_pca.best_params_}")
print(f"Najbolji CV score sa PCA: {grid_search_svm_pca.best_score_:.4f}")

# Poređenje sa i bez PCA
if grid_search_svm.best_score_ > grid_search_svm_pca.best_score_:
    print("SVM radi bolje BEZ PCA redukcije")
    final_svm_sentiment = best_svm
    use_pca_svm = False
else:
    print("SVM radi bolje SA PCA redukcijom")
    final_svm_sentiment = grid_search_svm_pca.best_estimator_
    use_pca_svm = True

## **Section 7:** Final testing on an independent test set

In [ ]:
print("\n" + "="*50)
print("FINALNO TESTIRANJE NA NEZAVISNOM TEST SKUPU")
print("="*50)

print("\n--- SENTIMENT ANALYSIS ---")

# Kombinuj train i validation za finalno treniranje
X_full_train = X_train + X_val
y_full_train = y_train + y_val

print(f"Finalni training set: {len(X_full_train)} samples")

In [ ]:
# 1. LSTM - retreniranje na celom skupu
tokenizer_final = Tokenizer(num_words=num_of_words,
                           filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
                           lower=True, split=' ')
tokenizer_final.fit_on_texts(X_full_train)

X_full_seq = tokenizer_final.texts_to_sequences(X_full_train)
X_test_seq_final = tokenizer_final.texts_to_sequences(X_final_test)

X_full_seq = pad_sequences(X_full_seq, max_length)
X_test_seq_final = pad_sequences(X_test_seq_final, max_length)

final_lstm = create_lstm_model(32, 16, 10)
final_lstm.fit(X_full_seq, np.array(y_full_train), epochs=10, batch_size=64, verbose=0)

# Predikcije i mere za LSTM
lstm_pred = (final_lstm.predict(X_test_seq_final) > 0.5).astype(int).flatten()
lstm_acc = accuracy_score(y_final_test, lstm_pred)
lstm_prec = precision_score(y_final_test, lstm_pred)
lstm_rec = recall_score(y_final_test, lstm_pred)
lstm_f1 = f1_score(y_final_test, lstm_pred)

print(f"\nLSTM rezultati:")
print(f"Accuracy: {lstm_acc:.4f}")
print(f"Precision: {lstm_prec:.4f}")
print(f"Recall: {lstm_rec:.4f}")
print(f"F1-score: {lstm_f1:.4f}")

In [ ]:
# 2. SVM - finalno treniranje
X_full_tfidf = tfidf_sentiment.fit_transform(X_full_train)
X_test_tfidf_final = tfidf_sentiment.transform(X_final_test)

if use_pca_svm:
    X_full_scaled = scaler.fit_transform(X_full_tfidf)
    X_test_scaled = scaler.transform(X_test_tfidf_final)
    X_full_pca = pca.fit_transform(X_full_scaled.toarray())
    X_test_pca = pca.transform(X_test_scaled.toarray())
    final_svm_sentiment.fit(X_full_pca, y_full_train)
    svm_pred = final_svm_sentiment.predict(X_test_pca)
else:
    final_svm_sentiment.fit(X_full_tfidf, y_full_train)
    svm_pred = final_svm_sentiment.predict(X_test_tfidf_final)

svm_acc = accuracy_score(y_final_test, svm_pred)
svm_prec = precision_score(y_final_test, svm_pred)
svm_rec = recall_score(y_final_test, svm_pred)
svm_f1 = f1_score(y_final_test, svm_pred)

print(f"\nSVM rezultati:")
print(f"Accuracy: {svm_acc:.4f}")
print(f"Precision: {svm_prec:.4f}")
print(f"Recall: {svm_rec:.4f}")
print(f"F1-score: {svm_f1:.4f}")

## **Section 8:** Data visualization

In [ ]:
print("\n--- Generisanje vizualizacija ---")

# Confusion Matrix za sentiment analizu
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_lstm = confusion_matrix(y_final_test, lstm_pred)
sns.heatmap(cm_lstm, annot=True, fmt='d', ax=axes[0], cmap='Blues')
axes[0].set_title('LSTM Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

cm_svm = confusion_matrix(y_final_test, svm_pred)
sns.heatmap(cm_svm, annot=True, fmt='d', ax=axes[1], cmap='Blues')
axes[1].set_title('SVM Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('segmentation_confusion_matrices.png')
plt.show()

In [ ]:
# ROC krive za sentiment analizu
plt.figure(figsize=(10, 6))

# LSTM ROC
lstm_proba = final_lstm.predict(X_test_seq_final).flatten()
fpr_lstm, tpr_lstm, _ = roc_curve(y_final_test, lstm_proba)
roc_auc_lstm = auc(fpr_lstm, tpr_lstm)

# SVM ROC
svm_proba = final_svm_sentiment.predict_proba(X_test_tfidf_final if not use_pca_svm else X_test_pca)[:, 1]
fpr_svm, tpr_svm, _ = roc_curve(y_final_test, svm_proba)
roc_auc_svm = auc(fpr_svm, tpr_svm)

plt.plot(fpr_lstm, tpr_lstm, label=f'LSTM (AUC = {roc_auc_lstm:.3f})')
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC = {roc_auc_svm:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Sentiment Analysis')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('roc_curves_sentiment.png')
plt.show()

## **Section 9:** Final results and saving the model

In [ ]:
# Poređenje algoritama za sentiment analizu
results_df = pd.DataFrame({
    'Algorithm': ['LSTM', 'SVM'],
    'Accuracy': [lstm_acc, svm_acc],
    'Precision': [lstm_prec, svm_prec],
    'Recall': [lstm_rec, svm_rec],
    'F1-score': [lstm_f1, svm_f1],
    'Dimensionality Reduction': ['None', 'PCA' if use_pca_svm else 'None']
})

print("\n" + "="*50)
print("FINALNI REZULTATI - SENTIMENT ANALYSIS")
print("="*50)
print(results_df.to_string(index=False))

In [ ]:
print("\n--- Čuvanje modela ---")
joblib.dump(final_svm_sentiment, 'final_svm_sentiment_model.pkl')
joblib.dump(tfidf_sentiment, 'tfidf_sentiment_vectorizer.pkl')
joblib.dump(pca, 'pca_transformer.pkl')
final_lstm.save('final_lstm_sentiment_model.h5')

print("Svi sentiment analysis modeli su sačuvani!")
print("- final_svm_sentiment_model.pkl")
print("- tfidf_sentiment_vectorizer.pkl") 
print("- pca_transformer.pkl")
print("- final_lstm_sentiment_model.h5")